# Personal AI Data Analyst Notebook

**Problem:** Data analysis can often be a complex, time-consuming, and iterative process, especially for users who might not be experts in programming or data science. Extracting insights, visualizing trends, and automating tasks often requires deep technical knowledge.

**Solution:** This notebook presents your very own **Personal AI Data Analyst**. It's designed to simplify your data exploration journey. By integrating a Streamlit application, it empowers you to:

1.  **Effortlessly Upload Data:** Bring in your CSV, Excel, or JSON files with ease.
2.  **Get Guided Analysis:** Receive intelligent suggestions for common data analysis tasks.
3.  **Automate Code Generation:** Transform your analysis ideas into executable Python code, either through predefined patterns or, optionally, with the help of a local Large Language Model (LLM).

This tool is built to make data analysis more accessible, efficient, and interactive for everyone.

## Setup

First, we need to install the necessary Python packages and optionally configure Ollama if you wish to use local LLM capabilities for custom prompts.

In [15]:
pip install streamlit pandas matplotlib numpy scipy openpyxl scikit-learn

### Install Required Libraries

This cell installs `streamlit`, `pandas`, `matplotlib`, `numpy`, `scipy`, and `openpyxl`. These are essential for running the Streamlit app and performing data analysis and visualization.

In [3]:
!ollama pull llama3.1

/bin/bash: line 1: ollama: command not found


### Optional: Ollama Setup

The `ollama pull llama3.1` command attempts to download the `llama3.1` model for local LLM inference. This is **optional**. If `ollama` is not installed on your system or you don't wish to use local LLMs, you can skip this cell or ensure the `Use local LLM` checkbox in the Streamlit app's sidebar is unchecked. The `ask_llm` function is designed to handle cases where Ollama is not available.

In [4]:
import io
import tempfile
import subprocess
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import textwrap
import sys
import os

# Optional duckdb import (not required)
try:
    import duckdb
except Exception:
    duckdb = None

# ----------------- Load data -----------------
def _looks_like_csv(raw_bytes: bytes) -> bool:
    try:
        sample = raw_bytes[:1024].decode(errors="ignore")
    except Exception:
        return False
    return "," in sample and "\n" in sample

def load_data(file_or_path) -> pd.DataFrame:
    """
    Accepts Streamlit UploadedFile, path string/Path, or file-like object.
    Returns pandas DataFrame.
    """
    if isinstance(file_or_path, (str, Path)):
        p = Path(file_or_path)
        s = p.suffix.lower()
        if s == ".csv":
            return pd.read_csv(p)
        if s in {".xls", ".xlsx"}:
            return pd.read_excel(p)
        if s == ".json":
            return pd.read_json(p)
        return pd.read_csv(p)

    # file-like (UploadedFile)
    name = getattr(file_or_path, "name", None)
    suffix = Path(name).suffix.lower() if name else None
    raw = file_or_path.read()
    if isinstance(raw, str):
        raw = raw.encode("utf-8")
    bio = io.BytesIO(raw)

    if suffix == ".csv" or (suffix is None and _looks_like_csv(raw)):
        bio.seek(0); return pd.read_csv(bio)
    if suffix in {".xls", ".xlsx"}:
        bio.seek(0); return pd.read_excel(bio)
    if suffix == ".json":
        bio.seek(0); return pd.read_json(bio)
    # fallback
    bio.seek(0)
    try:
        return pd.read_csv(bio)
    except Exception:
        bio.seek(0); return pd.read_json(bio)

### `load_data` Function

This function is responsible for loading data from various file types (CSV, Excel, JSON) into a Pandas DataFrame. It can handle Streamlit's `UploadedFile` objects, file paths, or file-like objects.

## Helper Functions

Below are several Python helper functions that facilitate data loading, prompt suggestions, conversion of prompts to code, execution of generated code, and interaction with an optional local LLM.

In [16]:
def _detect_column_types(df: pd.DataFrame):
    numeric = df.select_dtypes(include=[np.number]).columns.tolist()
    datetime = []
    # try to infer datetime columns
    for c in df.columns:
        if np.issubdtype(df[c].dtype, np.datetime64):
            datetime.append(c)
        else:
            # try to parse small sample as date
            try:
                sample = df[c].dropna().astype(str).iloc[:20]
                parsed = pd.to_datetime(sample, errors="coerce")
                if parsed.notna().sum() >= max(1, min(5, len(sample)//2)):
                    datetime.append(c)
            except Exception:
                pass
    # categoricals: low cardinality non-numeric
    categorical = [c for c in df.columns if c not in numeric + datetime and df[c].nunique(dropna=True) <= 50]
    return {"numeric": numeric, "datetime": datetime, "categorical": categorical}

def suggest_prompts(df: pd.DataFrame, max_suggestions: int = 8):
    """
    Return a list of helpful, ready-to-run prompt strings for the dataset.
    Deterministic and works without any LLM.
    """
    types = _detect_column_types(df)
    numeric = types["numeric"]
    datetime = types["datetime"]
    categorical = types["categorical"]

    suggestions = []
    # Basic summary
    suggestions.append("Summarize the dataset in 5 bullet points (rows, columns, missing values, numeric columns, top categorical).")
    # Top value queries
    if categorical:
        col = categorical[0]
        suggestions.append(f"Show the top 10 counts for the categorical column '{col}'.")
    # Numeric summaries
    if numeric:
        suggestions.append(f"Show summary statistics (count, mean, std, min, 25%, 50%, 75%, max) for numeric columns.")
        col = numeric[0]
        suggestions.append(f"Create a histogram of the numeric column '{col}'.")
        if len(numeric) >= 2:
            suggestions.append(f"Create a scatter plot comparing '{numeric[0]}' (x) vs '{numeric[1]}' (y).")
        suggestions.append(f"Show the top 10 rows sorted by '{col}' descending.")
    # Time series
    if datetime:
        dcol = datetime[0]
        # choose a numeric for aggregation if exists
        ag = numeric[0] if numeric else None
        if ag:
            suggestions.append(f"Create a time series of monthly sum of '{ag}' using the datetime column '{dcol}'.")
        else:
            suggestions.append(f"Show counts per month using the datetime column '{dcol}'.")
    # Correlation
    if len(numeric) >= 2:
        suggestions.append("Show the correlation matrix heatmap for numeric columns.")
    # Generic top-k
    suggestions.append("Find rows that look like anomalies using z-score > 3 on numeric columns and show top 20.")
    # Deep Learning / ML suggestion
    if numeric and categorical:
        suggestions.append(f"Train a simple neural network for classification of '{categorical[0]}' using numeric features.")
    # limit suggestions
    return suggestions[:max_suggestions]

### `_detect_column_types` and `suggest_prompts` Functions

-   `_detect_column_types`: Infers the data types (numeric, datetime, categorical) of columns in a DataFrame.
-   `suggest_prompts`: Generates a list of ready-to-run analysis prompts based on the detected column types. These prompts are designed to be deterministic and do not require an LLM.

In [6]:
def prompt_to_code(prompt: str, df: pd.DataFrame):
    """
    Convert known prompt templates into runnable python code strings.
    If the prompt is custom/unrecognized, return None (so UI can send to LLM instead).
    """
    p = prompt.strip().lower()

    # Summary
    if p.startswith("summarize the dataset"):
        code = textwrap.dedent("""
            # produce a short summary as printed text
            info = []
            info.append(f"Rows: {len(df)}, Columns: {len(df.columns)}")
            info.append("Column types: " + ", ".join([f\"{c}:{str(df[c].dtype)[:10]}\" for c in df.columns[:10]]))
            miss = df.isnull().sum().sort_values(ascending=False).head(10)
            info.append("Top missing: " + ", ".join([f\"{idx}:{val}\" for idx,val in miss.items() if val>0]))
            numeric = df.select_dtypes(include=['number']).columns.tolist()
            info.append(f\"Numeric columns count: {len(numeric)}\")
            # print concise bullets
            result = \"\\n\".join([\"- \"+i for i in info])
        """)
        return code

    # Top counts for categorical
    if "top 10 counts for the categorical column" in p or "top 10 counts" in p and "'" in p:
        # try to extract column name between quotes
        import re
        m = re.search(r"'([^']+)'", prompt)
        if not m:
            m = re.search(r'"([^"]+)"', prompt)
        col = m.group(1) if m else None
        if col:
            code = textwrap.dedent(f"""
                # top 10 counts for '{col}'
                result = df['{col}'].value_counts(dropna=False).head(10).reset_index()
                result.columns = ['value','count']
            """)
            return code

    # Summary statistics for numeric
    if "summary statistics" in p or "describe" in p:
        code = textwrap.dedent("""
            result = df.select_dtypes(include=['number']).describe().T
        """)
        return code

    # Histogram
    if p.startswith("create a histogram of the numeric column") or "histogram of the numeric column" in p:
        import re
        m = re.search(r"'([^']+)'", prompt)
        col = m.group(1) if m else None
        if col:
            code = textwrap.dedent(f"""
                # histogram for '{col}'
                plt.figure(figsize=(6,4))
                df['{col}'].dropna().astype(float).hist(bins=30)
                plt.title('Histogram of {col}')
                plt.xlabel('{col}')
                plt.ylabel('count')
                # produce an image by saving to result_img_path variable
                result_img_path = None
            """)
            # We'll return plotting code that uses plt; execution will save figure
            return code

    # Scatter plot
    if "scatter plot comparing" in p and "vs" in p:
        import re
        m = re.search(r"'([^']+)' \\(x\\) vs '([^']+)' \\(y\\)", prompt)
        if m:
            xcol, ycol = m.group(1), m.group(2)
            code = textwrap.dedent(f"""
                plt.figure(figsize=(6,4))
                df.plot.scatter(x='{xcol}', y='{ycol}')
                plt.title('{ycol} vs {xcol}')
                result_img_path = None
            """)
            return code

    # Top N rows sorted by col
    if p.startswith("show the top 10 rows sorted by"):
        import re
        m = re.search(r"by '([^']+)'", prompt)
        if m:
            col = m.group(1)
            code = textwrap.dedent(f"""
                result = df.sort_values('{col}', ascending=False).head(10).reset_index(drop=True)
            """)
            return code

    # Time series monthly sum
    if "monthly sum" in p and "using the datetime column" in p:
        import re
        m = re.search(r"sum of '([^']+)' using the datetime column '([^']+)'", prompt)
        if m:
            ag, dcol = m.group(1), m.group(2)
            code = textwrap.dedent(f"""
                tmp = df.copy()
                tmp['{dcol}'] = pd.to_datetime(tmp['{dcol}'], errors='coerce')
                res = tmp.dropna(subset=['{dcol}'])
                res = res.set_index('{dcol}').resample('M')['{ag}'].sum().reset_index()
                result = res
            """)
            return code

    # Counts per month (datetime only)
    if "counts per month using the datetime column" in p:
        import re
        m = re.search(r"datetime column '([^']+)'", prompt)
        dcol = m.group(1) if m else None
        if dcol:
            code = textwrap.dedent(f"""
                tmp = df.copy()
                tmp['{dcol}'] = pd.to_datetime(tmp['{dcol}'], errors='coerce')
                res = tmp.dropna(subset=['{dcol}']).set_index('{dcol}').resample('M').size().reset_index(name='count')
                result = res
            """)
            return code

    # Correlation heatmap
    if "correlation matrix heatmap" in p or "correlation heatmap" in p:
        code = textwrap.dedent("""
            corr = df.select_dtypes(include=['number']).corr()
            import matplotlib.pyplot as plt
            plt.figure(figsize=(6,5))
            plt.imshow(corr, cmap='viridis', aspect='auto')
            plt.colorbar()
            plt.xticks(range(len(corr)), corr.columns, rotation=90)
            plt.yticks(range(len(corr)), corr.columns)
            plt.title('Correlation matrix')
            result_img_path = None
        """)
        return code

    # Anomaly detection using z-score
    if "anomalies" in p and "z-score" in p:
        code = textwrap.dedent("""
            from scipy import stats
            num = df.select_dtypes(include=['number']).dropna()
            if num.shape[1]==0:
                result = pd.DataFrame()
            else:
                z = np.abs(stats.zscore(num.select_dtypes(include=['number'])))
                mask = (z > 3).any(axis=1)
                result = df.loc[mask].head(20).reset_index(drop=True)
        """)
        return code

    # Unknown / custom prompts -> return None
    return None

def prompt_to_code(prompt: str, df: pd.DataFrame):
    """
    Convert known prompt templates into runnable python code strings.
    If the prompt is custom/unrecognized, return None (so UI can send to LLM instead).
    """
    p = prompt.strip().lower()

    # Summary
    if p.startswith("summarize the dataset"):
        code = textwrap.dedent("""
            # produce a short summary as printed text
            info = []
            info.append(f"Rows: {len(df)}, Columns: {len(df.columns)}")
            info.append("Column types: " + ", ".join([f\"{c}:{str(df[c].dtype)[:10]}\" for c in df.columns[:10]])))
            miss = df.isnull().sum().sort_values(ascending=False).head(10)
            info.append("Top missing: " + ", ".join([f\"{idx}:{val}\" for idx,val in miss.items() if val>0])))
            numeric = df.select_dtypes(include=['number']).columns.tolist()
            info.append(f\"Numeric columns count: {len(numeric)}\")
            # print concise bullets
            result = "\\n".join(["- "+i for i in info])
        """)
        return code

    # Top counts for categorical
    if "top 10 counts for the categorical column" in p or "top 10 counts" in p and "'" in p:
        # try to extract column name between quotes
        import re
        m = re.search(r"'([^']+)'", prompt)
        if not m:
            m = re.search(r'"([^"]+)"', prompt)
        col = m.group(1) if m else None
        if col:
            code = textwrap.dedent(f"""
                # top 10 counts for '{col}'
                result = df['{col}'].value_counts(dropna=False).head(10).reset_index()
                result.columns = ['value','count']
            """)
            return code

    # Summary statistics for numeric
    if "summary statistics" in p or "describe" in p:
        code = textwrap.dedent("""
            result = df.select_dtypes(include=['number']).describe().T
        """)
        return code

    # Histogram
    if p.startswith("create a histogram of the numeric column") or "histogram of the numeric column" in p:
        import re
        m = re.search(r"'([^']+)'", prompt)
        col = m.group(1) if m else None
        if col:
            code = textwrap.dedent(f"""
                # histogram for '{col}'
                plt.figure(figsize=(6,4))
                df['{col}'].dropna().astype(float).hist(bins=30)
                plt.title('Histogram of {col}')
                plt.xlabel('{col}')
                plt.ylabel('count')
                # produce an image by saving to result_img_path variable
                result_img_path = None
            """)
            # We'll return plotting code that uses plt; execution will save figure
            return code

    # Scatter plot
    if "scatter plot comparing" in p and "vs" in p:
        import re
        m = re.search(r"'([^']+)' \(x\) vs '([^']+)' \(y\)", prompt)
        if m:
            xcol, ycol = m.group(1), m.group(2)
            code = textwrap.dedent(f"""
                plt.figure(figsize=(6,4))
                df.plot.scatter(x='{xcol}', y='{ycol}')
                plt.title('{ycol} vs {xcol}')
                result_img_path = None
            """)
            return code

    # Top N rows sorted by col
    if p.startswith("show the top 10 rows sorted by"):
        import re
        m = re.search(r"by '([^']+)'", prompt)
        if m:
            col = m.group(1)
            code = textwrap.dedent(f"""
                result = df.sort_values('{col}', ascending=False).head(10).reset_index(drop=True)
            """)
            return code

    # Time series monthly sum
    if "monthly sum" in p and "using the datetime column" in p:
        import re
        m = re.search(r"sum of '([^']+)' using the datetime column '([^']+)'", prompt)
        if m:
            ag, dcol = m.group(1), m.group(2)
            code = textwrap.dedent(f"""
                tmp = df.copy()
                tmp['{dcol}'] = pd.to_datetime(tmp['{dcol}'], errors='coerce')
                res = tmp.dropna(subset=['{dcol}'])
                res = res.set_index('{dcol}').resample('M')['{ag}'].sum().reset_index()
                result = res
            """)
            return code

    # Counts per month (datetime only)
    if "counts per month using the datetime column" in p:
        import re
        m = re.search(r"datetime column '([^']+)'", prompt)
        dcol = m.group(1) if m else None
        if dcol:
            code = textwrap.dedent(f"""
                tmp = df.copy()
                tmp['{dcol}'] = pd.to_datetime(tmp['{dcol}'], errors='coerce')
                res = tmp.dropna(subset=['{dcol}']).set_index('{dcol}').resample('M').size().reset_index(name='count')
                result = res
            """)
            return code

    # Correlation heatmap
    if "correlation matrix heatmap" in p or "correlation heatmap" in p:
        code = textwrap.dedent("""
            corr = df.select_dtypes(include=['number']).corr()
            import matplotlib.pyplot as plt
            plt.figure(figsize=(6,5))
            plt.imshow(corr, cmap='viridis', aspect='auto')
            plt.colorbar()
            plt.xticks(range(len(corr)), corr.columns, rotation=90)
            plt.yticks(range(len(corr)), corr.columns)
            plt.title('Correlation matrix')
            result_img_path = None
        """)
        return code

    # Anomaly detection using z-score
    if "anomalies" in p and "z-score" in p:
        code = textwrap.dedent("""
            from scipy import stats
            num = df.select_dtypes(include=['number']).dropna()
            if num.shape[1]==0:
                result = pd.DataFrame()
            else:
                z = np.abs(stats.zscore(num.select_dtypes(include=['number'])))
                mask = (z > 3).any(axis=1)
                result = df.loc[mask].head(20).reset_index(drop=True)
        """)
        return code

    # Deep Learning / MLP Classification
    if p.startswith("train a simple neural network for classification of") and "using numeric features" in p:
        import re
        m = re.search(r"classification of '([^']+)' using numeric features", prompt)
        target_col = m.group(1) if m else None
        if target_col and target_col in df.columns:
            code = textwrap.dedent(f"""
                from sklearn.model_selection import train_test_split
                from sklearn.neural_network import MLPClassifier
                from sklearn.metrics import classification_report
                from sklearn.preprocessing import LabelEncoder

                # Prepare data
                numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
                if '{target_col}' in numeric_features:
                    numeric_features.remove('{target_col}') # Remove target if it's numeric

                X = df[numeric_features].dropna()
                y_full = df.loc[X.index, '{target_col}']

                # Handle categorical target
                le = LabelEncoder()
                y = le.fit_transform(y_full)

                # Filter X to only include rows where y is not NaN (after LabelEncoder)
                # (LabelEncoder will raise error if NaN, so y_full is already clean here)

                if X.empty or len(y) != len(X):
                    result = "Error: Not enough valid data for training after dropping NaNs, or target column mismatch."
                else:
                    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

                    # Train MLP Classifier
                    mlp = MLPClassifier(hidden_layer_sizes=(10, 5), max_iter=1000, random_state=42)
                    mlp.fit(X_train, y_train)

                    # Evaluate
                    y_pred = mlp.predict(X_test)
                    report = classification_report(y_test, y_pred, target_names=le.classes_)
                    result = f"MLP Classification Report for '{target_col}':\\n" + report
            """)
            return code

    # Unknown / custom prompts -> return None
    return None

### `prompt_to_code` Function

This function is the core of the AI data analyst's "code generation" capability. It acts as a deterministic parser that translates specific natural language prompts into executable Python code snippets. Its primary goal is to handle common analysis requests efficiently without relying on an LLM for every standard task.

**Key features:**

-   **Template Matching:** It contains a series of `if` statements that check if the user's prompt matches a predefined template (e.g., "summarize the dataset", "create a histogram", "show correlation matrix").
-   **Dynamic Code Generation:** If a match is found, it generates a Python code string tailored to the specific request and the dataset's column names. It utilizes Pandas for data manipulation and Matplotlib for visualizations.
-   **Handling New Functionalities:** This function has been updated to include the logic for generating code for **simple neural network classification** tasks, specifically for `scikit-learn`'s `MLPClassifier`. When a prompt like "Train a simple neural network for classification of '[target_column]' using numeric features" is detected, it generates the necessary code for data preparation, model training, and evaluation.
-   **Fallback Mechanism:** If the prompt does not match any known template, the function returns `None`, signaling to the Streamlit application that the request is custom and might need to be routed to an external LLM (if enabled by the user).

In [7]:
def run_code(df: pd.DataFrame, code: str):
    """
    Execute code string in a restricted local namespace.
    Returns a dict:
      - {"type":"text","output":...}
      - {"type":"dataframe","df": pandas.DataFrame}
      - {"type":"image","path": path_to_png}
    Execution conventions:
      - If code sets a variable `result` to a DataFrame or string, we return it.
      - If code uses matplotlib to plot, we save the current figure to a temp PNG and return image.
      - If code raises, return error text.
    """
    # prepare namespace
    local_ns = {"pd": pd, "np": np, "df": df, "plt": plt}
    # capture prints
    old_stdout = sys.stdout
    stdout_buf = io.StringIO()
    sys.stdout = stdout_buf
    try:
        # run
        exec(code, {}, local_ns)
        # first, if plotting occurred (plt has a current figure), save it
        # If code created result_img_path variable, prefer it
        if "result_img_path" in local_ns and local_ns["result_img_path"]:
            path = local_ns["result_img_path"]
            return {"type": "image", "path": path}
        # check for figure in plt
        figs = plt.get_fignums()
        if figs:
            with tempfile.NamedTemporaryFile(delete=False, suffix=".png") as f:
                plt.savefig(f.name, bbox_inches="tight", dpi=150)
                plt.close("all")
                return {"type": "image", "path": f.name}
        # check for result variable
        if "result" in local_ns:
            res = local_ns["result"]
            if isinstance(res, pd.DataFrame):
                return {"type": "dataframe", "df": res}
            else:
                return {"type": "text", "output": str(res)}
        # otherwise, return captured stdout
        out = stdout_buf.getvalue().strip()
        if out:
            return {"type": "text", "output": out}
        return {"type": "text", "output": "Execution finished. No result produced."}
    except Exception as e:
        return {"type": "text", "output": f"Execution error: {e}"}
    finally:
        sys.stdout = old_stdout

### `run_code` Function

Executes a given Python code string within a restricted local namespace. It captures printed output, can return a DataFrame, or save a Matplotlib plot to a temporary image file. It also handles execution errors.

In [8]:
def ask_llm(prompt: str, model: str = "llama3.1", timeout: int = 60) -> str:
    """
    Send prompt to local Ollama via CLI. Returns stdout text.
    If ollama is not installed or fails, returns an error string starting with [LLM...].
    Expect the model to return code inside ```python blocks.
    """
    try:
        proc = subprocess.run(["ollama", "run", model], input=prompt.encode("utf-8"),
                              stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout)
        out = proc.stdout.decode("utf-8", errors="replace")
        err = proc.stderr.decode("utf-8", errors="replace")
        if not out and err:
            return f"[LLM-error] {err}"
        return out
    except FileNotFoundError:
        return "[LLM-missing] ollama not found on PATH."
    except Exception as e:
        return f"[LLM-failed] {e}"

### `ask_llm` Function

This function is designed to send a prompt to a local Ollama instance via its CLI. It returns the LLM's response, which is expected to contain Python code within a ````python ```` block. It includes error handling for cases where Ollama is not installed or fails to respond.

### Structuring Helper Functions for Modularity

To improve code modularity and maintainability, helper functions can be organized in several ways:

1.  **Grouping by Concern:** Functions related to a specific task (e.g., data loading, data cleaning, visualization) can be grouped together. This notebook already follows this by having `load_data` related to data loading, and `_detect_column_types`, `suggest_prompts`, `prompt_to_code`, `run_code`, and `ask_llm` related to prompt processing and code execution.
2.  **Separate Files/Modules:** For larger projects, these helper functions could be placed in separate Python files (e.g., `data_utils.py`, `llm_utils.py`). These modules could then be imported into the main notebook or application (`app.py`). This keeps the notebook cleaner and allows for easier reuse of code across different projects.
3.  **Classes:** If a set of functions operates on a common piece of data or shares state, encapsulating them within a class can be beneficial. For example, a `DataProcessor` class could contain methods for loading, cleaning, and transforming data.

For this notebook, given its scope, defining functions directly in the cells under a clear 'Helper Functions' section is a practical and readable approach. As the project scales, moving towards separate modules would be the next logical step.

### New Deep Learning Functionality: Simple MLP Classifier

To expand the analytical capabilities of your Personal AI Data Analyst, we've integrated a basic deep learning functionality: a simple Multilayer Perceptron (MLP) classifier using `scikit-learn`.

**How it works:**

1.  **Installation:** `scikit-learn` has been added to the initial `pip install` command.
2.  **Prompt Suggestion:** The `suggest_prompts` function now intelligently offers a prompt like: `"Train a simple neural network for classification of '{categorical_column}' using numeric features."` This suggestion appears if your dataset contains both numeric and categorical columns, making it suitable for a classification task.
3.  **Code Generation (`prompt_to_code`):** When you select or type a prompt matching the pattern for a simple neural network classification, the `prompt_to_code` function will generate Python code to perform the following steps:
    *   **Data Preparation:** It identifies numeric features as input (`X`) and the specified categorical column as the target (`y`). It also handles `NaN` values by dropping rows that contain them in the selected features and target.
    *   **Label Encoding:** The categorical target variable is converted into numerical labels using `LabelEncoder`.
    *   **Train-Test Split:** The data is split into training and testing sets (80% train, 20% test).
    *   **Model Training:** A `MLPClassifier` with two hidden layers (10 and 5 neurons) is initialized and trained on the training data.
    *   **Evaluation:** The trained model is evaluated on the test set, and a detailed `classification_report` is generated, showing precision, recall, f1-score, and support for each class.

This allows you to quickly get a basic deep learning model up and running for classification tasks directly from a natural language prompt, providing initial insights into potential predictive relationships within your data.

### Brief Explanation: Multilayer Perceptron (MLP) Classifier

The Multilayer Perceptron (MLP) Classifier is a fundamental type of artificial neural network used for classification tasks. It's often referred to as a "feedforward" neural network because information flows in only one direction—forward—from the input nodes, through the hidden nodes (if any), and to the output nodes.

**How it works (Simplified):**

1.  **Input Layer:** Receives the raw data (our numeric features in this case).
2.  **Hidden Layers:** These are where the "learning" happens. Each node in a hidden layer takes inputs from the previous layer, applies a set of weights and a mathematical function (activation function), and passes the result to the next layer. The network learns patterns and relationships in the data by adjusting these weights during training.
3.  **Output Layer:** Produces the final prediction. For classification, this layer typically assigns probabilities to each possible class (e.g., predicting the category of a categorical column).

**Training Process:**

The MLP learns by comparing its predictions with the actual target values and adjusting its internal weights to minimize the difference (error). This process is repeated many times over the dataset, allowing the network to gradually improve its predictive accuracy. `scikit-learn`'s `MLPClassifier` efficiently handles this training, making it accessible for basic deep learning applications.

In [11]:
import streamlit as st
import pandas as pd

st.set_page_config(page_title="Personal AI Data Analyst", layout="wide")
st.title("🧠 Personal AI Data Analyst — Interactive Dashboard")

st.sidebar.header("Settings")
use_llm = st.sidebar.checkbox("Use local LLM (ollama) for custom prompts", value=False)
llm_model = st.sidebar.text_input("LLM model name (ollama)", value="llama3.1")
st.sidebar.markdown("If you don't have `ollama` installed, leave this off and use built-in prompts.")

uploaded = st.file_uploader("Upload CSV, Excel, or JSON", type=["csv","xls","xlsx","json"])

df = None # Initialize df to None

if uploaded is None:
    st.info("Upload a CSV / XLSX / JSON to get started. Suggestions will appear automatically.")
else:
    # Load data
    try:
        df = load_data(uploaded)
        st.success("File loaded.")
    except Exception as e:
        st.error(f"Failed to load file: {e}")

if df is not None: # Proceed only if df has been successfully loaded
    with st.expander("Preview data (first 100 rows)"):
        st.dataframe(df.head(100))

    # Generate suggestions
    suggestions = suggest_prompts(df)
    st.markdown("## Suggested analyses (pick one or write your own)")
    col1, col2 = st.columns([3,1])
    with col1:
        selected = st.selectbox("Choose a suggested prompt", options=suggestions)
        custom = st.text_area("Or write a custom prompt (leave blank to use the selected suggestion)", height=80)
    with col2:
        st.markdown("**Quick actions**")
        if st.button("Show suggestions again"):
            st.write(suggestions)

    # Determine final prompt
    final_prompt = custom.strip() if custom and custom.strip() else selected

    st.markdown("### Final prompt")
    st.write(final_prompt)

    # Run button
    if st.button("Run analysis"):
        with st.spinner("Running..."):
            code = None # Initialize code and res
            res = None

            # First try deterministic conversion
            code = prompt_to_code(final_prompt, df)
            if code:
                res = run_code(df, code)
            else:
                # No deterministic code found. If user requested LLM, send the prompt.
                if use_llm:
                    system = (
                        "You are a helpful data analyst and will respond with Python code only.\n"
                        "You must return code inside a ```python ... ``` block. The DataFrame is named `df`.\n"
                        "Use pandas for data manipulation and matplotlib for charts. Do not import heavy libs.\n"
                        "If returning a chart, produce matplotlib code that draws the figure (no show()) and nothing else.\n"
                    )
                    raw = system + "\n# User prompt: " + final_prompt
                    llm_out = ask_llm(raw, model=llm_model)
                    if llm_out.startswith("[LLM-missing]") or llm_out.startswith("[LLM-"):
                        st.warning("LLM unavailable or returned an error. Falling back to built-in behavior is not possible for this custom prompt.")
                        st.write(llm_out)
                    elif "```python" in llm_out:
                        try:
                            code = llm_out.split("```python")[1].split("```")[0]
                            res = run_code(df, code)
                        except Exception as e:
                            st.error(f"Failed to execute code from LLM: {e}")
                            st.write(llm_out)
                    else:
                        st.error("LLM did not return a python code block. Showing raw LLM output:")
                        st.write(llm_out)
                else:
                    st.error("This is a custom prompt that the app cannot deterministically convert to code. Enable 'Use local LLM' in the sidebar to let a local model generate Python, or edit your prompt to match one of the suggested patterns.")

        # Display result
        if res is not None:
            if res["type"] == "text":
                st.markdown("#### Output (text)")
                st.text(res["output"])
            elif res["type"] == "dataframe":
                st.markdown("#### Output (table)")
                st.dataframe(res["df"])
                csv = res["df"].to_csv(index=False).encode("utf-8")
                st.download_button("Download result as CSV", data=csv, file_name="result.csv", mime="text/csv")
            elif res["type"] == "image":
                st.markdown("#### Output (chart)")
                st.image(res["path"], use_column_width=True)
            else:
                st.write("Unknown result type", res)

2026-07-26 20:09:44.357 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-26 20:09:44.359 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-26 20:09:44.362 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-26 20:09:44.364 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-26 20:09:44.368 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-26 20:09:44.370 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-26 20:09:44.373 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-26 20:09:44.375 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [13]:
!streamlit run app.py

Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


## Run the Streamlit Application

After ensuring that `app.py` has been written to the file system (by running the previous cell), you can execute the Streamlit application using the command below. Streamlit will provide a URL (usually `localhost:8501` or a Colab-specific public URL) where you can access the interactive dashboard.

In [17]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import io
import tempfile
import subprocess
from pathlib import Path
import textwrap
import sys

# Helper functions (copied from the notebook's global scope)

def load_data(file_or_path) -> pd.DataFrame:
    """
    Accepts Streamlit UploadedFile, path string/Path, or file-like object.
    Returns pandas DataFrame.
    """
    def _looks_like_csv(raw_bytes: bytes) -> bool:
        try:
            sample = raw_bytes[:1024].decode(errors="ignore")
        except Exception:
            return False
        return "," in sample and "\n" in sample

    if isinstance(file_or_path, (str, Path)):
        p = Path(file_or_path)
        s = p.suffix.lower()
        if s == ".csv":
            return pd.read_csv(p)
        if s in {".xls", ".xlsx"}:
            return pd.read_excel(p)
        if s == ".json":
            return pd.read_json(p)
        return pd.read_csv(p)

    # file-like (UploadedFile)
    name = getattr(file_or_path, "name", None)
    suffix = Path(name).suffix.lower() if name else None
    raw = file_or_path.read()
    if isinstance(raw, str):
        raw = raw.encode("utf-8")
    bio = io.BytesIO(raw)

    if suffix == ".csv" or (suffix is None and _looks_like_csv(raw)):
        bio.seek(0); return pd.read_csv(bio)
    if suffix in {".xls", ".xlsx"}:
        bio.seek(0); return pd.read_excel(bio)
    if suffix == ".json":
        bio.seek(0); return pd.read_json(bio)
    # fallback
    bio.seek(0)
    try:
        return pd.read_csv(bio)
    except Exception:
        bio.seek(0); return pd.read_json(bio)

def _detect_column_types(df: pd.DataFrame):
    numeric = df.select_dtypes(include=[np.number]).columns.tolist()
    datetime = []
    # try to infer datetime columns
    for c in df.columns:
        if np.issubdtype(df[c].dtype, np.datetime64):
            datetime.append(c)
        else:
            # try to parse small sample as date
            try:
                sample = df[c].dropna().astype(str).iloc[:20]
                parsed = pd.to_datetime(sample, errors="coerce")
                if parsed.notna().sum() >= max(1, min(5, len(sample)//2)):
                    datetime.append(c)
            except Exception:
                pass
    # categoricals: low cardinality non-numeric
    categorical = [c for c in df.columns if c not in numeric + datetime and df[c].nunique(dropna=True) <= 50]
    return {"numeric": numeric, "datetime": datetime, "categorical": categorical}

def suggest_prompts(df: pd.DataFrame, max_suggestions: int = 8):
    """
    Return a list of helpful, ready-to-run prompt strings for the dataset.
    Deterministic and works without any LLM.
    """
    types = _detect_column_types(df)
    numeric = types["numeric"]
    datetime = types["datetime"]
    categorical = types["categorical"]

    suggestions = []
    # Basic summary
    suggestions.append("Summarize the dataset in 5 bullet points (rows, columns, missing values, numeric columns, top categorical).")
    # Top value queries
    if categorical:
        col = categorical[0]
        suggestions.append(f"Show the top 10 counts for the categorical column '{col}'.")
    # Numeric summaries
    if numeric:
        suggestions.append(f"Show summary statistics (count, mean, std, min, 25%, 50%, 75%, max) for numeric columns.")
        col = numeric[0]
        suggestions.append(f"Create a histogram of the numeric column '{col}'.")
        if len(numeric) >= 2:
            suggestions.append(f"Create a scatter plot comparing '{numeric[0]}' (x) vs '{numeric[1]}' (y).")
        suggestions.append(f"Show the top 10 rows sorted by '{col}' descending.")
    # Time series
    if datetime:
        dcol = datetime[0]
        # choose a numeric for aggregation if exists
        ag = numeric[0] if numeric else None
        if ag:
            suggestions.append(f"Create a time series of monthly sum of '{ag}' using the datetime column '{dcol}'.")
        else:
            suggestions.append(f"Show counts per month using the datetime column '{dcol}'.")
    # Correlation
    if len(numeric) >= 2:
        suggestions.append("Show the correlation matrix heatmap for numeric columns.")
    # Generic top-k
    suggestions.append("Find rows that look like anomalies using z-score > 3 on numeric columns and show top 20.")
    # Deep Learning / ML suggestion
    if numeric and categorical:
        suggestions.append(f"Train a simple neural network for classification of '{categorical[0]}' using numeric features.")
    # limit suggestions
    return suggestions[:max_suggestions]

def prompt_to_code(prompt: str, df: pd.DataFrame):
    """
    Convert known prompt templates into runnable python code strings.
    If the prompt is custom/unrecognized, return None (so UI can send to LLM instead).
    """
    p = prompt.strip().lower()

    # Summary
    if p.startswith("summarize the dataset"):
        code = textwrap.dedent("""
            # produce a short summary as printed text
            info = []
            info.append(f"Rows: {len(df)}, Columns: {len(df.columns)}")
            info.append("Column types: " + ", ".join([f\"{c}:{str(df[c].dtype)[:10]}\" for c in df.columns[:10]])))
            miss = df.isnull().sum().sort_values(ascending=False).head(10)
            info.append("Top missing: " + ", ".join([f\"{idx}:{val}\" for idx,val in miss.items() if val>0])))
            numeric = df.select_dtypes(include=['number']).columns.tolist()
            info.append(f\"Numeric columns count: {len(numeric)}\")
            # print concise bullets
            result = "\\n".join(["- "+i for i in info])
        """)
        return code

    # Top counts for categorical
    if "top 10 counts for the categorical column" in p or "top 10 counts" in p and "'" in p:
        # try to extract column name between quotes
        import re
        m = re.search(r"'([^']+)'", prompt)
        if not m:
            m = re.search(r'"([^"]+)"', prompt)
        col = m.group(1) if m else None
        if col:
            code = textwrap.dedent(f"""
                # top 10 counts for '{col}'
                result = df['{col}'].value_counts(dropna=False).head(10).reset_index()
                result.columns = ['value','count']
            """)
            return code

    # Summary statistics for numeric
    if "summary statistics" in p or "describe" in p:
        code = textwrap.dedent("""
            result = df.select_dtypes(include=['number']).describe().T
        """)
        return code

    # Histogram
    if p.startswith("create a histogram of the numeric column") or "histogram of the numeric column" in p:
        import re
        m = re.search(r"'([^']+)'", prompt)
        col = m.group(1) if m else None
        if col:
            code = textwrap.dedent(f"""
                # histogram for '{col}'
                plt.figure(figsize=(6,4))
                df['{col}'].dropna().astype(float).hist(bins=30)
                plt.title('Histogram of {col}')
                plt.xlabel('{col}')
                plt.ylabel('count')
                # produce an image by saving to result_img_path variable
                result_img_path = None
            """)
            # We'll return plotting code that uses plt; execution will save figure
            return code

    # Scatter plot
    if "scatter plot comparing" in p and "vs" in p:
        import re
        m = re.search(r"'([^']+)' \(x\) vs '([^']+)' \(y\)", prompt)
        if m:
            xcol, ycol = m.group(1), m.group(2)
            code = textwrap.dedent(f"""
                plt.figure(figsize=(6,4))
                df.plot.scatter(x='{xcol}', y='{ycol}')
                plt.title('{ycol} vs {xcol}')
                result_img_path = None
            """)
            return code

    # Top N rows sorted by col
    if p.startswith("show the top 10 rows sorted by"):
        import re
        m = re.search(r"by '([^']+)'", prompt)
        if m:
            col = m.group(1)
            code = textwrap.dedent(f"""
                result = df.sort_values('{col}', ascending=False).head(10).reset_index(drop=True)
            """)
            return code

    # Time series monthly sum
    if "monthly sum" in p and "using the datetime column" in p:
        import re
        m = re.search(r"sum of '([^']+)' using the datetime column '([^']+)'", prompt)
        if m:
            ag, dcol = m.group(1), m.group(2)
            code = textwrap.dedent(f"""
                tmp = df.copy()
                tmp['{dcol}'] = pd.to_datetime(tmp['{dcol}'], errors='coerce')
                res = tmp.dropna(subset=['{dcol}'])
                res = res.set_index('{dcol}').resample('M')['{ag}'].sum().reset_index()
                result = res
            """)
            return code

    # Counts per month (datetime only)
    if "counts per month using the datetime column" in p:
        import re
        m = re.search(r"datetime column '([^']+)'", prompt)
        dcol = m.group(1) if m else None
        if dcol:
            code = textwrap.dedent(f"""
                tmp = df.copy()
                tmp['{dcol}'] = pd.to_datetime(tmp['{dcol}'], errors='coerce')
                res = tmp.dropna(subset=['{dcol}']).set_index('{dcol}').resample('M').size().reset_index(name='count')
                result = res
            """)
            return code

    # Correlation heatmap
    if "correlation matrix heatmap" in p or "correlation heatmap" in p:
        code = textwrap.dedent("""
            corr = df.select_dtypes(include=['number']).corr()
            import matplotlib.pyplot as plt
            plt.figure(figsize=(6,5))
            plt.imshow(corr, cmap='viridis', aspect='auto')
            plt.colorbar()
            plt.xticks(range(len(corr)), corr.columns, rotation=90)
            plt.yticks(range(len(corr)), corr.columns)
            plt.title('Correlation matrix')
            result_img_path = None
        """)
        return code

    # Anomaly detection using z-score
    if "anomalies" in p and "z-score" in p:
        code = textwrap.dedent("""
            from scipy import stats
            num = df.select_dtypes(include=['number']).dropna()
            if num.shape[1]==0:
                result = pd.DataFrame()
            else:
                z = np.abs(stats.zscore(num.select_dtypes(include=['number'])))
                mask = (z > 3).any(axis=1)
                result = df.loc[mask].head(20).reset_index(drop=True)
        """)
        return code

    # Deep Learning / MLP Classification
    if p.startswith("train a simple neural network for classification of") and "using numeric features" in p:
        import re
        m = re.search(r"classification of '([^']+)' using numeric features", prompt)
        target_col = m.group(1) if m else None
        if target_col and target_col in df.columns:
            code = textwrap.dedent(f"""
                from sklearn.model_selection import train_test_split
                from sklearn.neural_network import MLPClassifier
                from sklearn.metrics import classification_report
                from sklearn.preprocessing import LabelEncoder

                # Prepare data
                numeric_features = df.select_dtypes(include=[np.number]).columns.tolist()
                if '{target_col}' in numeric_features:
                    numeric_features.remove('{target_col}') # Remove target if it's numeric

                X = df[numeric_features].dropna()
                y_full = df.loc[X.index, '{target_col}']

                # Handle categorical target
                le = LabelEncoder()
                y = le.fit_transform(y_full)

                # Filter X to only include rows where y is not NaN (after LabelEncoder)
                # (LabelEncoder will raise error if NaN, so y_full is already clean here)

                if X.empty or len(y) != len(X):
                    result = "Error: Not enough valid data for training after dropping NaNs, or target column mismatch."
                else:
                    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

                    # Train MLP Classifier
                    mlp = MLPClassifier(hidden_layer_sizes=(10, 5), max_iter=1000, random_state=42)
                    mlp.fit(X_train, y_train)

                    # Evaluate
                    y_pred = mlp.predict(X_test)
                    report = classification_report(y_test, y_pred, target_names=le.classes_)
                    result = f"MLP Classification Report for '{target_col}':\\n" + report
            """)
            return code

    # Unknown / custom prompts -> return None
    return None

def run_code(df: pd.DataFrame, code: str):
    """
    Execute code string in a restricted local namespace.
    Returns a dict:
      - {"type":"text","output":...}
      - {"type":"dataframe","df": pandas.DataFrame}
      - {"type":"image","path": path_to_png}
    Execution conventions:
      - If code sets a variable `result` to a DataFrame or string, we return it.
      - If code uses matplotlib to plot, we save the current figure to a temp PNG and return image.
      - If code raises, return error text.
    """
    # prepare namespace
    local_ns = {"pd": pd, "np": np, "df": df, "plt": plt}
    # capture prints
    old_stdout = sys.stdout
    stdout_buf = io.StringIO()
    sys.stdout = stdout_buf
    try:
        # run
        exec(code, {}, local_ns)
        # first, if plotting occurred (plt has a current figure), save it
        # If code created result_img_path variable, prefer it
        if "result_img_path" in local_ns and local_ns["result_img_path"]:
            path = local_ns["result_img_path"]
            return {"type": "image", "path": path}
        # check for figure in plt
        figs = plt.get_fignums()
        if figs:
            with tempfile.NamedTemporaryFile(delete=False, suffix=".png") as f:
                plt.savefig(f.name, bbox_inches="tight", dpi=150)
                plt.close("all")
                return {"type": "image", "path": f.name}
        # check for result variable
        if "result" in local_ns:
            res = local_ns["result"]
            if isinstance(res, pd.DataFrame):
                return {"type": "dataframe", "df": res}
            else:
                return {"type": "text", "output": str(res)}
        # otherwise, return captured stdout
        out = stdout_buf.getvalue().strip()
        if out:
            return {"type": "text", "output": out}
        return {"type": "text", "output": "Execution finished. No result produced."}
    except Exception as e:
        return {"type": "text", "output": f"Execution error: {e}"}
    finally:
        sys.stdout = old_stdout

def ask_llm(prompt: str, model: str = "llama3.1", timeout: int = 60) -> str:
    """
    Send prompt to local Ollama via CLI. Returns stdout text.
    If ollama is not installed or fails, returns an error string starting with [LLM...].
    Expect the model to return code inside ```python blocks.
    """
    try:
        proc = subprocess.run(["ollama", "run", model], input=prompt.encode("utf-8"),
                              stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout)
        out = proc.stdout.decode("utf-8", errors="replace")
        err = proc.stderr.decode("utf-8", errors="replace")
        if not out and err:
            return f"[LLM-error] {err}"
        return out
    except FileNotFoundError:
        return "[LLM-missing] ollama not found on PATH."
    except Exception as e:
        return f"[LLM-failed] {e}"


st.set_page_config(page_title="Personal AI Data Analyst", layout="wide")
st.title("🧠 Personal AI Data Analyst — Interactive Dashboard")

st.sidebar.header("Settings")
use_llm = st.sidebar.checkbox("Use local LLM (ollama) for custom prompts", value=False)
llm_model = st.sidebar.text_input("LLM model name (ollama)", value="llama3.1")
st.sidebar.markdown("If you don't have `ollama` installed, leave this off and use built-in prompts.")

uploaded = st.file_uploader("Upload CSV, Excel, or JSON", type=["csv","xls","xlsx","json"])

df = None # Initialize df to None

if uploaded is None:
    st.info("Upload a CSV / XLSX / JSON to get started. Suggestions will appear automatically.")
else:
    # Load data
    try:
        df = load_data(uploaded)
        st.success("File loaded.")
    except Exception as e:
        st.error(f"Failed to load file: {e}")

if df is not None: # Proceed only if df has been successfully loaded
    with st.expander("Preview data (first 100 rows)"):
        st.dataframe(df.head(100))

    # Generate suggestions
    suggestions = suggest_prompts(df)
    st.markdown("## Suggested analyses (pick one or write your own)")
    col1, col2 = st.columns([3,1])
    with col1:
        selected = st.selectbox("Choose a suggested prompt", options=suggestions)
        custom = st.text_area("Or write a custom prompt (leave blank to use the selected suggestion)", height=80)
    with col2:
        st.markdown("**Quick actions**")
        if st.button("Show suggestions again"):
            st.write(suggestions)

    # Determine final prompt
    final_prompt = custom.strip() if custom and custom.strip() else selected

    st.markdown("### Final prompt")
    st.write(final_prompt)

    # Run button
    if st.button("Run analysis"):
        with st.spinner("Running..."):
            code = None # Initialize code and res
            res = None

            # First try deterministic conversion
            code = prompt_to_code(final_prompt, df)
            if code:
                res = run_code(df, code)
            else:
                # No deterministic code found. If user requested LLM, send the prompt.
                if use_llm:
                    system = (
                        "You are a helpful data analyst and will respond with Python code only.\n"
                        "You must return code inside a ```python ... ``` block. The DataFrame is named `df`.\n"
                        "Use pandas for data manipulation and matplotlib for charts. Do not import heavy libs.\n"
                        "If returning a chart, produce matplotlib code that draws the figure (no show()) and nothing else.\n"
                    )
                    raw = system + "\n# User prompt: " + final_prompt
                    llm_out = ask_llm(raw, model=llm_model)
                    if llm_out.startswith("[LLM-missing]") or llm_out.startswith("[LLM-"):
                        st.warning("LLM unavailable or returned an error. Falling back to built-in behavior is not possible for this custom prompt.")
                        st.write(llm_out)
                    elif "```python" in llm_out:
                        try:
                            code = llm_out.split("```python")[1].split("```")[0]
                            res = run_code(df, code)
                        except Exception as e:
                            st.error(f"Failed to execute code from LLM: {e}")
                            st.write(llm_out)
                    else:
                        st.error("LLM did not return a python code block. Showing raw LLM output:")
                        st.write(llm_out)
                else:
                    st.error("This is a custom prompt that the app cannot deterministically convert to code. Enable 'Use local LLM' in the sidebar to let a local model generate Python, or edit your prompt to match one of the suggested patterns.")

        # Display result
        if res is not None:
            if res["type"] == "text":
                st.markdown("#### Output (text)")
                st.text(res["output"])
            elif res["type"] == "dataframe":
                st.markdown("#### Output (table)")
                st.dataframe(res["df"])
                csv = res["df"].to_csv(index=False).encode("utf-8")
                st.download_button("Download result as CSV", data=csv, file_name="result.csv", mime="text/csv")
            elif res["type"] == "image":
                st.markdown("#### Output (chart)")
                st.image(res["path"], use_column_width=True)
            else:
                st.write("Unknown result type", res)

Overwriting app.py


In [18]:
!streamlit run app.py



2026-07-26 20:15:38.820 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.90.90.53:8501

  Stopping...
  Stopping...
  Stopping...
  Stopping...


## Streamlit Application (`app.py`)

**Problem:** While helper functions streamline backend logic, presenting a user-friendly interface for data interaction remains a challenge. Manually running code cells for each analysis step can be cumbersome and less intuitive.

**Solution:** This cell writes the interactive Streamlit application code into a file named `app.py`. The `%%writefile` magic command is used for this purpose. This `app.py` file contains the logic for a web-based dashboard that directly addresses the problem by providing:

-   **Intuitive File Upload:** A simple interface to upload your datasets.
-   **Guided Prompt Selection:** A curated list of analysis prompts to kickstart your exploration.
-   **Seamless Analysis Execution:** A click-button interface to run analyses and display results (tables, text, or charts).
-   **Optional LLM Integration:** The ability to use a local LLM for generating custom analysis code from your unique prompts.

This centralized, interactive dashboard enhances user experience and makes the powerful analysis tools accessible without needing to interact directly with the Python code.

## Project Workflow: Personal AI Data Analyst

This project aims to simplify complex data analysis through an interactive Streamlit application, leveraging both deterministic code generation and optional LLM integration.

### 1. Problem Statement & Solution
*   **Problem:** Traditional data analysis is often complex, time-consuming, and requires specialized programming skills.
*   **Solution:** The **Personal AI Data Analyst** notebook provides an accessible, interactive platform to streamline data exploration and analysis for users of all skill levels.

### 2. Environment Setup & Dependencies
*   **Core Libraries:** Installation of essential Python packages including `streamlit`, `pandas`, `matplotlib`, `numpy`, `scipy`, `openpyxl`, and `scikit-learn` for data handling, visualization, and machine learning.
*   **Optional LLM Setup:** Provision for integrating local Large Language Models (LLMs) via Ollama, offering flexibility for custom analysis prompts.

### 3. Core Helper Function Development
*   **`load_data`:** Robust function for loading various data formats (CSV, Excel, JSON) into pandas DataFrames.
*   **`_detect_column_types`:** Utility to infer and categorize column types (numeric, datetime, categorical) for intelligent analysis.
*   **`suggest_prompts`:** Generates context-aware analysis prompt suggestions based on dataset characteristics, including deep learning tasks.
*   **`prompt_to_code`:** Translates natural language prompts into executable Python code snippets, covering standard analyses and advanced tasks like `MLPClassifier` training.
*   **`run_code`:** Executes generated Python code in a controlled environment, capturing and presenting results as text, DataFrames, or Matplotlib plots.
*   **`ask_llm`:** Facilitates communication with a local Ollama instance to generate code for custom, unrecognized prompts.

### 4. Deep Learning Integration
*   **MLP Classifier:** Integrated `scikit-learn`'s `MLPClassifier` for basic deep learning classification tasks, enabling users to train simple neural networks with a natural language prompt.

### 5. Streamlit Application (`app.py`) Creation
*   **Interactive Dashboard:** A Streamlit application (`app.py`) was developed to provide a user-friendly interface for:
    *   Uploading data files.
    *   Selecting or entering analysis prompts.
    *   Running analyses and displaying results intuitively.
*   **Modularity & Iteration:** The `app.py` was iteratively developed and regenerated, incorporating helper functions and debugging issues (`ModuleNotFoundError`, `NameError`, `SyntaxError`) to ensure a robust and functional application.

### 6. Application Execution
*   **`!streamlit run app.py`:** Command used to launch the interactive Streamlit dashboard, making the AI Data Analyst accessible via a web interface.